In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:


def clean_missing(df,s1='mean',s2='most_frequent'):
    import pandas as pd
    
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.compose import ColumnTransformer
    df_missing_values = df.isnull().sum()
    df_numeric_columns = df.select_dtypes(include=["int64","float64"]).keys()
    columns_numeric_missing = [var for var in df_numeric_columns if df_missing_values[var]>0]
    
    df_categorical_columns = df.select_dtypes(include=["object"]).keys()
    columns_categorical_missing = [var for var in df_categorical_columns if df_missing_values[var]>0]
    
    numeric_value_mean_imputer = Pipeline(steps=[("imputer", SimpleImputer(strategy=s1))])
    
    if s2=='constant':
        x=input("what do you want to fill in place of missing values")
        categorical_value_mode_imputer = Pipeline(steps=[("imputer", SimpleImputer(strategy=s2,fill_value=x))])
       # s2=strategy='constant',fill_value= x
    else:     
        categorical_value_mode_imputer = Pipeline(steps=[("imputer", SimpleImputer(strategy=s2))])
    
    
    preprocessing = ColumnTransformer(transformers=[("mean_imputer", numeric_value_mean_imputer, columns_numeric_missing),
                                                ("mode_imputer", categorical_value_mode_imputer, columns_categorical_missing)])
    
    #scale data
    df_clean_null_value = preprocessing.fit_transform(df)

    df_missing_value_solve = pd.DataFrame(df_clean_null_value, columns=columns_numeric_missing+columns_categorical_missing)

    df.update(df_missing_value_solve)
    return df

In [ ]:
df=pd.read_csv("/kaggle/input/ml-club-nits/train.csv")
df1=pd.read_csv("/kaggle/input/ml-club-nits/test.csv")

In [ ]:
clean_missing(df)
df

In [ ]:
df.info()

#No label encoding
y = df["Dataset"]
X = df.drop('Dataset',axis=1)

from  sklearn.preprocessing import LabelEncoder
ly=LabelEncoder()


y1=ly.fit_transform(df['Gender'])
y2=pd.DataFrame(y1,columns=['Gender'])

X= X.drop(["Gender",'ID'],axis=1)

X=pd.concat([X,y2],axis=1)
X


In [ ]:

y=df['Dataset']

df=df.drop(['ID','Dataset'],axis=1)
cc=['Gender']

dummy=pd.get_dummies(df[cc])

df=df.drop(cc,axis=1)

X=pd.concat([df,dummy],axis=1)


X.head()


In [ ]:
from sklearn.preprocessing import MinMaxScaler
sc_x = MinMaxScaler()

X1 = sc_x.fit_transform(X)
X=pd.DataFrame(columns=X.columns,data=X1)

X




In [ ]:

from sklearn.ensemble import StackingClassifier

from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB



from sklearn.model_selection  import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30,stratify=y,random_state = 5)


model_1 =GaussianNB()
model_2 = svm.SVC()
model_3 = RandomForestClassifier()
model_4=  DecisionTreeClassifier(random_state=0)
model_5=KNeighborsClassifier(n_neighbors=2)



#model_1.fit(X_train,y_train)
#print(model_1.score(X_test,y_test))


#model_2.fit(X_train,y_train)
#print(model_2.score(X_test,y_test))

#model_3.fit(X_train,y_train)
#print(model_3.score(X_test,y_test))

#model_4.fit(X_train,y_train)
#print(model_4.score(X_test,y_test))

#model_5.fit(X_train,y_train)
#print(model_5.score(X_test,y_test))



estimators = [ ('m1', model_1),
   ('m2', model_2),('m3',model_3 ),('m4',model_4)]

#stacking_classifier sc
sc = StackingClassifier(   estimators=estimators, final_estimator=model_3 )


#step 3
sc.fit(X_train,y_train)



#step4

sc.score(X_test,y_test)

# very poor score .lets try adaboost(boosting)

In [ ]:
# importing utility modul# importing utility modules
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# importing machine learning models for prediction
from sklearn.ensemble import GradientBoostingRegressor

# importing voting classifer
from sklearn.ensemble import VotingClassifier



# Splitting between train data into training and validation dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,stratify=y, test_size=0.20)
  
# initializing the boosting module with default parameters
boosting = GradientBoostingRegressor()
  
# training the model on the train dataset
boosting.fit(X_train, y_train)
  
# predicting the output on the test dataset
pred_final = boosting.predict(X_test)
  
# printing the root mean squared error between real value and predicted value
#print(mean_squared_error(y_test, pred_final))
boosting.score(X_train, y_train)


# now after getting poor score ,lets try voting

In [ ]:
# importing utility modules
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

# importing machine learning models for prediction
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm

from sklearn.linear_model import LogisticRegression

# importing voting classifer
from sklearn.ensemble import VotingClassifier




# Splitting between train data into training and validation dataset
X_train, X_test, y_train, y_test = train_test_split(X, y,stratify=y ,test_size=0.30)

# initializing all the model objects with default parameters


model_1 =GaussianNB()
model_2 = svm.SVC()
model_3 = RandomForestClassifier()
model_4=  DecisionTreeClassifier(random_state=0)
model_5=KNeighborsClassifier(n_neighbors=2)

# Making the final model using voting classifier
voting = VotingClassifier(
	estimators=[ ('m1', model_1),
   ('m2', model_2),('m3',model_3 ),('m4',model_4),('m5',model_5)], voting='hard')

# training all the model on the train dataset
voting.fit(X_train, y_train)

# predicting the output on the test dataset
pred_final = voting.predict(X_test)

# printing log loss between actual and predicted value
#print(log_loss(y_test, pred_final))
voting.score(X_test,y_test)


In [ ]:
model_1.fit(X_train,y_train)
#print(model_1.score(X_test,y_test))

model_2.fit(X_train,y_train)
#print(model_2.score(X_test,y_test))

model_3.fit(X_train,y_train)
#print(model_3.score(X_test,y_test))

model_4.fit(X_train,y_train)
#print(model_4.score(X_test,y_test))

model_5.fit(X_train,y_train)
#print(model_5.score(X_test,y_test))

In [ ]:
#sc
#boosting
#voting
pred=sc.predict(X_test)
from sklearn.metrics import accuracy_score
print('accuracy:%s'%accuracy_score(y_test,pred))

In [ ]:
X_train

# now predict the test set of competition by the model we made (ensemble/combined model of 5 models)

In [ ]:
df1=pd.read_csv("/kaggle/input/ml-club-nits/test.csv")
clean_missing(df1)
df1

In [ ]:
df1.info()

In [ ]:
ID=df1['ID']
df1=df1.drop(['ID'],axis=1)
cc=['Gender']

dummy=pd.get_dummies(df1[cc])

df1=df1.drop(cc,axis=1)

X1=pd.concat([df1,dummy],axis=1)


X1.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
sc_x = MinMaxScaler()

X2 = sc_x.fit_transform(X1)
X3=pd.DataFrame(columns=X.columns,data=X2)

X3

In [ ]:
pred=sc.predict(X3)

In [ ]:
pred

In [ ]:
pred1=pd.DataFrame(data=pred,columns=['Dataset'])

In [ ]:
pred1

In [ ]:
ID

In [ ]:
df1=pd.concat([ID,pred1],axis=1)

In [ ]:
df1

In [ ]:
#df1.to_csv('Liver_disease_estimation_final.csv')

# miscelleneous: calculate acc, precision ,recall ,f1 score of all model(because output is binary/logistic regression)

In [ ]:
model_1.fit(X_train,y_train)
#print(model_1.score(X_test,y_test))

model_2.fit(X_train,y_train)
#print(model_2.score(X_test,y_test))

model_3.fit(X_train,y_train)
#print(model_3.score(X_test,y_test))

model_4.fit(X_train,y_train)
#print(model_4.score(X_test,y_test))

model_5.fit(X_train,y_train)
#print(model_5.score(X_test,y_test))

pred=sc.predict(X_test)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(classification_report(y_test,pred))

In [ ]:
pred=model_1.predict(X_test)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(classification_report(y_test,pred))

In [ ]:
pred=model_2.predict(X_test)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(classification_report(y_test,pred))

In [ ]:
pred=model_3.predict(X_test)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(classification_report(y_test,pred))

In [ ]:
pred=model_4.predict(X_test)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(classification_report(y_test,pred))

In [ ]:
pred=model_5.predict(X_test)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
print(classification_report(y_test,pred))

# Exporting stacking model (sc)

In [ ]:

import pickle
  
# Save the trained model as a pickle string.
filename='sc.sav'
saved_model = pickle.dump(sc,open(filename,'wb'))



# Save Model Using Pickle
import pandas
from sklearn import model_selection
from sklearn.linear_model import LogisticRegression
import pickle
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = pandas.read_csv(url, names=names)
array = dataframe.values
X = array[:,0:8]
Y = array[:,8]
test_size = 0.33
seed = 7
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(X, Y, test_size=test_size, random_state=seed)
# Fit the model on training set
model = LogisticRegression()
model.fit(X_train, Y_train)
# save the model to disk
filename = 'finalized_model.sav'
pickle.dump(model, open(filename, 'wb'))

# some time later...

# load the model from disk
loaded_model = pickle.load(open(filename, 'rb'))
result = loaded_model.score(X_test, Y_test)
print(result)

In [ ]:
# Load libraries
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import datasets
# Import train_test_split function
from sklearn.model_selection import train_test_split
#Import scikit-learn metrics module for accuracy calculation
from sklearn import metrics

In [ ]:
X_train

In [ ]:
# Create adaboost classifer object
abc = AdaBoostClassifier(n_estimators=40,learning_rate=1)
# Train Adaboost Classifer
model = abc.fit(X_train, y_train)

cl=DecisionTreeClassifier()
cl.fit(X_train, y_train)
#Predict the response for test dataset
y_pred = model.predict(X_test)

In [ ]:
#load dataset and clean
# Split dataset into training set and test set
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3) # 70% training and 30% test
print(model.score(X_test,y_test))
print(cl.score(X_test,y_test))


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.linear_model import LogisticRegression


from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor



model_1 =GaussianNB()
model_2 = svm.SVC()
model_3 = RandomForestClassifier()
model_4=  DecisionTreeClassifier(random_state=0)
model_5=KNeighborsClassifier(n_neighbors=2)
model_6 = LinearRegression()
model_7 = RandomForestRegressor()




model_1.fit(X_train,y_train)
#print(model_1.score(X_test,y_test))

model_2.fit(X_train,y_train)
#print(model_2.score(X_test,y_test))

model_3.fit(X_train,y_train)
#print(model_3.score(X_test,y_test))

model_4.fit(X_train,y_train)
#print(model_4.score(X_test,y_test))

model_5.fit(X_train,y_train)
#print(model_5.score(X_test,y_test))

model_6.fit(X_train,y_train)
model_7.fit(X_train,y_train)